In [1]:
# -*- coding: utf-8 -*-
"""
@author: Etienne Kras
"""

# generic imports
import sys
import os
import numpy as np
from pathlib import Path
import platform
import geopandas as gpd
import pandas as pd
import math
import time
import geemap
import json
import geojson
import ee

project = "bathymetry"
ee.Initialize(project=project) # use the GEE project-id here

# specific imports
from typing import Any, Dict, List, Optional
from geojson import Feature, FeatureCollection, dump
from shapely.geometry import Polygon, MultiPolygon, shape
from dateutil.relativedelta import *
from google.cloud import storage
from logging import Logger, getLogger
from googleapiclient.discovery import build
from re import sub
from ctypes import ArgumentError
from functools import partial
from dateutil.parser import parse

# custom functionality import without requirement to pip install package
local_path = r"C:\Users\kras\Documents\GitHub\ee-packages-py"  # path to local GitHub clone
sys.path.append(local_path)
from eepackages.applications.bathymetry import Bathymetry
from eepackages import tiler

logger: Logger = getLogger(__name__)

C:\Users\kras\AppData\Local\Temp\ipykernel_25284\4082885255.py:12: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


In [2]:
# TODO: 
# look if scale / crs does not influence the output used before exporting as we have differences between the GEE export and the local post-processed export

# Project specific toggles

In [3]:
# acknowledgements & code references:
# https://github.com/openearth/eo-bathymetry/
# https://github.com/openearth/eo-bathymetry-functions/
# https://github.com/gee-community/ee-packages-py

In [4]:
# see scheme at https://github.com/openearth/eo-bathymetry/blob/master/notebooks/rws-bathymetry/acces_api.pdf for a workflow visualization 

sdb_folder = r"11209821-cmems-global-sdb"

# project toggles
if platform.system().startswith("Windows"):
    main_fol = Path(r"p:\\") / sdb_folder
else:
    main_fol = Path("/mnt/p") / sdb_folder  # name of the main local folder 
bucket = "cmems-sdb" # name of the Google Cloud Storage bucket to store files in the cloud
credential_file = main_fol / "00_miscellaneous" / "KEYS" / "bathymetry-543b622ddce7.json" # Cloud Storage credential key
output_fol = r"01_intertidal/02_data/04_calibrated" # name of the overall project
aoi_fol = r"00_miscellaneous/AOI_testdyn"#AOI_upscale" #AOIs
mask_fol = r"00_miscellaneous/Feasibility_maps" # masks
gtsm_fol = r"01_intertidal/02_data/02_gtsm_files" # GTSM files
box_fol = r"00_miscellaneous/AOI_polygons_testdyn" # AOI_polygons # AOI_polygons_upscale
prog_fol = r"00_miscellaneous/progress_files" # saved progress (task) pickle files
project_name = "AOI_NL" # name of the project AoI, or one in the folder
draw_AoI = 0 # toggle 1 to draw AoI, 0 to load

# composite image toggles
mode = "intertidal_improved_100m_testdyn"#_upscaled" # specify mode, either "intertidal" or "subtidal"
start_date = "2021-01-01" # start date of the composites
stop_date = "2022-01-01" # end date of the composites
compo_int = 12 # composite interval [months]
compo_len = 12 # composite length [months]
scale = 100  # output resolution of the image [m]
GEBCO_scale = 465 #m, actually 463 m but rounded up to the nearest 5 m 
crs = "EPSG:4326" # output projection of the image

# tiling options
zoomed_list = [9, 10, 11] # list with zoom levels to be inspected
sel_tile = 1 # idx of chosen tile level in zoomed_list (inspect the map to chose it accordingly), z9 too big for in memory computations
# note, see https://www.openearth.nl/rws-bathymetry/2019.html; Z9 is optimal size..

# load google credentials, if specified
if not credential_file == "":  
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(credential_file)

# load GTSM & gebco data
gtsm_col = ee.FeatureCollection('projects/bathymetry/assets/gtsm_waterlevels')
gebco_image = ee.Image('projects/bathymetry/assets/gebco_2023_hat_lat')

# Pre-processing using the API

In [5]:
# draw or load Area of Interest (AoI)

# TODO: take center of AOI input file if present, put in random coordinate and let user find a place and draw a polygon
# TODO: fix horizontal tiling error (DOS) in API (to use multiple tiles) or move to single polygon run if AoI crosses multiple tiles
Map = geemap.Map(center=(54.2, 6.7), zoom=8) # initialize map with base in Hudayriat

if draw_AoI == 1:
    print("Please draw a polygon somewhere in a water body") # identifier
if draw_AoI == 0:
    # open AoI
    print("Loading and visualizing AoI") #identifier
    #AoIee = geemap.geojson_to_ee(os.path.join(main_fol,'AOI',project_name+'.geojson'))

    with open(main_fol / aoi_fol / (project_name + ".geojson"), 'r') as f:
        contents = geojson.loads(f.read())
    AoIee = ee.Geometry.MultiPolygon(contents["features"][0]["geometry"]["coordinates"])

    Map.addLayer(AoIee, {}, "AoI")

Map # show map

Loading and visualizing AoI


Map(center=[54.2, 6.7], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…

In [6]:
# (re)construct the AoI

if draw_AoI == 1:
    
    print("Constructing AoI from drawn polygon") # identifier
    
    # get AoI 
    AoIee = ee.FeatureCollection(Map.draw_features) # make featurecollection
    AoI = Polygon(AoIee.getInfo()["features"][0]["geometry"]["coordinates"][0]) # create AoI shapefile

    # export AoI
    features = []
    features.append(Feature(geometry=AoI, properties={"AoI": project_name}))
    AoIjson = FeatureCollection(features)
    with open(main_fol / "AOI" / (project_name + ".geojson"), "w") as f: # geojson
        dump(AoIjson, f)
    gdr = gpd.GeoDataFrame({"properties":{"AoI": project_name}, "geometry": AoI}, crs="EPSG:4326") #shp
    gdr.to_file(main_fol / "AOI" / (project_name+".shp"))
    bounds = ee.Geometry.Polygon([[[a,b] for a, b in zip(*AoI.exterior.coords.xy)]])
    
if draw_AoI == 0:
    print("Reconstructing AoI from loaded file")
    # get AoI
    with open(main_fol / aoi_fol / (project_name+".geojson")) as f:
        AoIjson = geojson.load(f)
    # try: # drawn polygon in this script
    #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"]) 
    # except: # drawn in QGIS / ArcGIS and written to geojson there (client file)
    #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"][0])
    # bounds = ee.Geometry.Polygon([[[a,b] for a, b in zip(*AoI.exterior.coords.xy)]])
    bounds = ee.Geometry.MultiPolygon(AoIjson["features"][0]["geometry"]["coordinates"])

    # make list of multipolygons a single one
    # MPL = []
    # for i in AoIjson["features"]:
    #     for j in i["geometry"]["coordinates"]:
    #         MPL.append(j)

    # bounds = ee.Geometry.MultiPolygon(MPL)

Reconstructing AoI from loaded file


# Retrieving the global intertidal processing area (mask)

In [7]:
# make the AoI a gdf dataframe (takes about 5 min), using geojson
gdf_AOI = gpd.GeoDataFrame.from_features(AoIjson)

# retrieve the (buffered) global coastal intertidal mask (geojson)
# allfiles = os.listdir(os.path.join(main_fol, mask_fol, "Buffered\AOI_results"))
# dfs = []
# for i in allfiles:
#     for j in os.listdir(os.path.join(main_fol, mask_fol, "Buffered\AOI_results", i)):
#         if "result.geojson" in j:
#             print(j) 
#             #open geojson
#             with open(os.path.join(main_fol, mask_fol, "Buffered\AOI_results", i, j)) as f:
#                 #data = geojson.load(f)
#                 df = gpd.read_file(f)

#                 # inner join to match the AOI and mask
#                 df_joined = df.sjoin(gdf_AOI, how="inner")

#                 dfs.append(df_joined) # append to list

# # concat the geodfs list
# gdfs = gpd.GeoDataFrame(pd.concat(dfs, ignore_index=True)) 

In [8]:
# retrieve the (buffered) global coastal intertidal mask (geojson), using parquet
allfiles = os.listdir(main_fol / mask_fol / "2024")#r"2023\Buffered\AOI_results")
for i in allfiles:
    if '.parquet' in i and "minus2" in i:
        print(i)
        # read parquet file
        df = gpd.read_parquet(main_fol / mask_fol / "2024" / i)#r"2023\Buffered\AOI_results" / i)

        # filter the data (3 = original, 2 = L-W mask, 1 = <HAT >LAT pixel)
        df_org = df[df["pixel_value"] == 3.0] # select only the pixels with value 3.0 (original) --> 415 k rows (polygons)

        # inner join to match the AOI and mask
        gdfs = df_org.sjoin(gdf_AOI, how="inner")

        # Intersection of polygons to the bounding box of the tile
        cropped_gdfs = gpd.overlay(gdfs, gdf_AOI, how='intersection')
        ce_gdfs = cropped_gdfs[["geometry"]].explode(index_parts=False) # explode potential multipols to pols

gebco_2024_latminus2_merge_result.parquet


In [9]:
# make a multipolygon
gdfs_mp = gpd.GeoDataFrame({'geometry': [MultiPolygon(list(ce_gdfs.geometry))]})

# make a ee.Geometry.MultiPolygon to have bounds
data = json.loads(gdfs_mp.to_json())
feature_collection = geojson.FeatureCollection(data['features'])
bounds = ee.Geometry.MultiPolygon(feature_collection["features"][0]["geometry"]["coordinates"])

# add to map
Map.addLayer(bounds, {}, "intertidal feasibility mask")

# Tiling the AoI

In [10]:
# function for tiling the AoI and showing it on the map
def add_tile_bounds(zoom):
    tiled = tiler.get_tiles_for_geometry(bounds, zoom)
    Map.addLayer(tiled.style(width=max(1, 10 - zoom), fillColor= "00000022"), {}, "tiles " + str(zoom))

    return(tiled)

In [11]:
# tiling the AoI for all zoom levels show on map to decide zoom level to go for
tiles = list(map(add_tile_bounds, zoomed_list)) # add tiles for different zoom levels

# tiling without showing it on the map
#tiles = list(tiler.get_tiles_for_geometry(bounds, zoom) for zoom in zoomed_list)

In [12]:
# tiling the AoI, show on map for one zoom level
#tiled = list(map(add_tile_bounds, [zoomed_list[sel_tile]]))

# selecting the correct tiled AoI without showing it on a map 
# note, adjust sel_tile accordingly 
tiled = [tiles[sel_tile]]

In [13]:
tiled[0]

In [14]:
# tiling multiple AoIs (in the input folder) without showing it on a map 
# note, adjust sel_tile accordingly 

allfiles = os.listdir(os.path.join(main_fol, aoi_fol)) #AOIs
tiled = []
file_list = []
for i in allfiles:
    if i.endswith("geojson"): #and not "CMEMS" in i and not "adjusted" in i: #and ("GER" in i or "BRA" in i or "ZAF" in i):
        print(i)
        with open(os.path.join(main_fol, aoi_fol, i)) as f:
            AoIjson = geojson.load(f)
        # try: # drawn polygon in this script
        #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"]) 
        # except: # drawn in QGIS / ArcGIS and written to geojson there (client file)
        #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"][0])
        # bounds = ee.Geometry.Polygon([[[a,b] for a, b in zip(*AoI.exterior.coords.xy)]])
        bounds = ee.Geometry.MultiPolygon(AoIjson["features"][0]["geometry"]["coordinates"])

        if "intertidal" in mode:
            # make the AoI a gdf dataframe (takes about 5 min)
            gdf_AOI = gpd.GeoDataFrame.from_features(AoIjson)

            # retrieve the (buffered) global coastal intertidal mask (geojson), using parquet
            maskfiles = os.listdir(main_fol / mask_fol / "2024")#r"2023\Buffered\AOI_results")
            for j in maskfiles:
                if '.parquet' in j and "minus2" in j:
                    print(j)
                    # read parquet file
                    df = gpd.read_parquet(main_fol / mask_fol / "2024" / j)#r"2023\Buffered\AOI_results" / j)

                    # filter the data (3 = original, 2 = L-W mask, 1 = <HAT >LAT pixel)
                    df_org = df[df["pixel_value"] == 3.0] # select only the pixels with value 3.0 (original) --> 415 k rows (polygons)

                    # inner join to match the AOI and mask
                    gdfs = df_org.sjoin(gdf_AOI, how="inner")

                    # Intersection of polygons to the bounding box of the tile
                    cropped_gdfs = gpd.overlay(gdfs, gdf_AOI, how='intersection')
                    ce_gdfs = cropped_gdfs[["geometry"]].explode(index_parts=False) # explode potential multipols to pols

                    # make a multipolygon
                    gdfs_mp = gpd.GeoDataFrame({'geometry': [MultiPolygon(list(ce_gdfs.geometry))]})

                    # make a ee.Geometry.MultiPolygon to have bounds
                    data = json.loads(gdfs_mp.to_json())
                    feature_collection = geojson.FeatureCollection(data['features'])
                    bounds = ee.Geometry.MultiPolygon(feature_collection["features"][0]["geometry"]["coordinates"])

        tiled.append(tiler.get_tiles_for_geometry(bounds, zoomed_list[sel_tile]))
        file_list.append(i)

AOI_AUS.geojson
gebco_2024_latminus2_merge_result.parquet
AOI_NL.geojson
gebco_2024_latminus2_merge_result.parquet


In [15]:
tiled[0]

In [16]:
# export selected tiles (represented by zoom level) to geojsons and build a dataframe
# note, adjust sel_tile accordingly 

# TODO: make folder AOI_polygons_%s automatically
ref_regions = gpd.read_file(os.path.join(main_fol / "00_miscellaneous" / "IPCC_regions", "IPCC-WGI-reference-regions-v4.geojson"))

df_boxes = pd.DataFrame() # this one is similar to what we construct for the world except from the 8 column onwards (AOI column is more here)
for tiles, file in zip(tiled, file_list):
    for idx, tile in enumerate(tiles.getInfo()["features"]):
        tile_pol = Polygon(tile['geometry']['coordinates'][0]) # create tile polygon
        tile_name = "z%s_x%s_y%s"%(tile["properties"]["zoom"], int(float(tile["properties"]["tx"])), int(float(tile["properties"]["ty"])))

        features = []
        features.append(Feature(geometry=tile_pol, properties={"name": tile_name, "id":tile['id'], "tx":float(tile["properties"]["tx"]), "ty":float(tile["properties"]["ty"]), "zoom":int(float(tile["properties"]["zoom"]))}))
        feature_collection = FeatureCollection(features)
        feature_collection.crs = {"type": "name","properties": {"name": "epsg:3857"}} # default EE projection
        with open(main_fol / box_fol / (tile_name + ".geojson"), "w") as f: # geojson
            dump(feature_collection, f)

        # make a df
        df = gpd.GeoDataFrame.from_features(feature_collection)
        df["path"] = main_fol / box_fol / (tile_name + ".geojson")
        df["AOI"] = file

        # merge in
        df_boxes = pd.concat([df_boxes, df], ignore_index=True)  

# match with ref regions
df_boxes = gpd.overlay(df_boxes, ref_regions.to_crs("EPSG:3857"), how='intersection')
df_boxes.drop(["id_2", "Continent", "Type", "Name"], axis=1, inplace=True)
df_boxes.rename(columns={"id_1":"id", "Acronym":"ref_region"}, inplace=True)

In [17]:
# append area percentages to the dataframe
df_boxes["area_perc_org"] = None # Create a new column in the existing GeoDataFrame
df_boxes["area_perc_red"] = None # Create a new column in the existing GeoDataFrame
for index, row in df_boxes.iterrows():

    # convert geoseries to gpd dataframe and set crs
    box_geom = gpd.GeoDataFrame(geometry=gpd.GeoSeries(row["geometry"]))
    box_geom.set_crs("EPSG:3857", inplace=True)

    # Perform spatial join are areas intersecting with the mask (as pre-filter)
    joined_gdf = df_org.sjoin(box_geom.to_crs("EPSG:4326"), how='inner', predicate='intersects') 
                    
    # Intersection of polygons to the bounding box of the tile
    cropped_gdf = gpd.overlay(joined_gdf, box_geom.to_crs("EPSG:4326"), how='intersection')

    # morphological erosion-dilation on the cropped_gdf
    cero = cropped_gdf.buffer(-GEBCO_scale/2/111120) # divided by 2 to have half eroded on 2 sides (makes 1 in total) & fix for unequal buffer in EPSG3857: https://www.reddit.com/r/QGIS/comments/oo1jgh/buffer_points_on_epsg_4326_wgs_84/
    cdil = cero.buffer(GEBCO_scale/2/111120) # get it back to the original (note empty geoemtries won't)
    cdil = cdil[~cdil.is_empty] # removing the empty geometries

    # make example plot
    # ax = joined_gdf.plot()
    # box_geom.to_crs("epsg:4326").plot(ax=ax, linewidth=0.5, edgecolor='black', facecolor='none')
    # cropped_gdf.plot(ax=ax, facecolor='red')
    # cero.plot(ax=ax, facecolor='green')
    # cdil.plot(ax=ax, facecolor='orange')

    # calculate the area
    df_boxes.loc[index, "area_perc_org"] = round((sum(cropped_gdf.geometry.area)/box_geom.to_crs("epsg:4326").area[0])*100,2) # percentage of the area covered by the mask
    df_boxes.loc[index, "area_perc_red"] = round((sum(cdil.area)/box_geom.to_crs("epsg:4326").area[0])*100,2) # percentage of the area covered by the mask

In [18]:
# append gtsm info to the dataframe

# Load GTSM stations
gdf_stations = gpd.read_file(main_fol / gtsm_fol / "gtsm_stations.geojson") # EPSG4326 standard

# Create a kd-tree for the stations
from scipy.spatial import cKDTree
tree = cKDTree(gdf_stations['geometry'].to_crs("EPSG:3857").apply(lambda x: (x.x, x.y)).tolist())

# Find the nearest station and its wrongfull (!) distance 
def find_nearest_station(row):
    dist, idx = tree.query((row['geometry'].centroid.x, row['geometry'].centroid.y)) 
    return pd.Series([int(idx), dist/1000, gdf_stations['geometry'][idx]]) # dist in km

# Find the nearest station
df_boxes[['idx_nearest_station', 'distance_nearest_station', 'nearest_station_coord']] = df_boxes.apply(find_nearest_station, axis=1)
df_boxes.set_crs("EPSG:3857", inplace=True)
df_boxes.to_crs("EPSG:4326", inplace=True)

# correct the wrongfull distance
def haversine_corr(tile):
    """
    Calculate the great-circle distance between two points on the Earth's surface using the Haversine formula.
    """
    # access the data
    lat1, lon1 = df_boxes[df_boxes["name"] == tile].geometry.centroid.iloc[0].y, df_boxes[df_boxes["name"] == tile].geometry.centroid.iloc[0].x  # Berlin
    lat2, lon2 = df_boxes[df_boxes["name"] == tile].nearest_station_coord.iloc[0].y, df_boxes[df_boxes["name"] == tile].nearest_station_coord.iloc[0].x   # Paris

    R = 6371  # Earth's radius in km

    # Convert latitude and longitude from degrees to radians
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)

    # Haversine formula
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c * 1000 # Distance in m

for idx, tile in enumerate(df_boxes["name"]):
    #print("%s of %s"%(idx, len(df_boxes["name"])), flush=True, end="\r")
    df_boxes.loc[df_boxes["name"] == tile, "distance_nearest_station_corr"] = float(haversine_corr(tile))

# replace boxes dist with corrected distance
df_boxes = df_boxes.drop(columns=['distance_nearest_station'])
df_boxes = df_boxes.rename(columns={"distance_nearest_station_corr": "distance_nearest_station"})
df_boxes.to_crs("EPSG:3857", inplace=True)

In [20]:
# # optional: checking consistency with the world file

# # open the world file
# file_path_boxes_csv = os.path.join(main_fol, '00_miscellaneous', 'AOI_polygons_world', 'df_boxes_world_dist_Z{}.csv'.format(zoomed_list[sel_tile]))
# df_allboxes = pd.read_csv(file_path_boxes_csv, index_col=0) # EPSG3857 standard
# df_allboxes_run = df_allboxes[df_allboxes["name"].isin(df_boxes["name"])]
# df_allboxes_run

# # sorting the global df
# df_allboxes_run = df_allboxes_run.sort_values(by=["name"])
# df_allboxes_run = df_allboxes_run.sort_index(axis=1)
# df_allboxes_run = df_allboxes_run.drop(columns=["id", "path", "geometry"])
# df_allboxes_run = df_allboxes_run.reset_index(drop=True)

# # sorting the construct AOI df
# df_boxes_sorted = df_boxes.sort_values(by=["name"])
# df_boxes_sorted = df_boxes_sorted.sort_index(axis=1)
# df_boxes_sorted = df_boxes_sorted.drop(columns=["AOI", "id", "path", "geometry"])
# df_boxes_sorted = df_boxes_sorted.reset_index(drop=True)

# # Check if the sorted DataFrames are identical, 
# # TODO: actual comparison with true or false somehow does not work
# print(df_boxes_sorted)
# print(df_allboxes_run)

# Compute SDB using the API

In [21]:
# functionality for intertidal sdb to pre-filter tiles to be processed

# filter tiles close to GTSM
def filter_tiles_close_to_GTSM(gtsm_col, start_date, stop_date, tile=ee.Feature(None),
                              max_spatial_offset=1):
    
    # Get area around the tile
    #tile_centroid = ee.Geometry.centroid(tile.geometry(), maxError=1)
    tile_footprint = ee.Geometry(tile.geometry())
    tile_buffer = tile_footprint.buffer(max_spatial_offset*1000)

    # Get period around image time
    tile_time_start = ee.Date(start_date)
    tile_time_end = ee.Date(stop_date) 
    
    # Filter gtsm data based on tile footprint and period
    gtsm_col = gtsm_col.filterBounds(tile_buffer)
    gtsm_col = gtsm_col.filterDate(ee.Date(tile_time_start.millis()), ee.Date(tile_time_end.millis()))

    # Add GTSM size to image
    tile = tile.set("max_gtsm_size", gtsm_col.size())
    
    return tile

In [22]:
# functions to compute sub & intertidal bathymetry proxies based on standardized SlippyMap tiling practice
# functions taken from: https://github.com/openearth/eo-bathymetry/blob/master/notebooks/rws-bathymetry/export_bathymetry.ipynb
# resembles similar behaviour as in https://github.com/openearth/eo-bathymetry-functions but slightly adjusted for local study 

def get_tile_subtidal_bathymetry(tile: ee.Feature, start: ee.String, stop: ee.String) -> ee.Image:
    """
    Get subtidal bathymetry based on tile geometry.
    Server-side compliant for GEE.

    args:
        tile (ee.Feature): tile geometry used to obtain bathymetry.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
    
    returns:
        ee.Image: image containing subtidal bathymetry covering tile.
    """

    bounds: ee.Geometry = ee.Feature(tile).geometry().bounds(1)
    sdb: Bathymetry = Bathymetry()
    zoom: ee.String = ee.String(tile.get("zoom"))
    tx: ee.String = ee.String(tile.get("tx"))
    ty: ee.String = ee.String(tile.get("ty"))
    tile_name: ee.String = ee.String("z").cat(zoom).cat("_x").cat(tx).cat("_y").cat(ty).replace("\.\d+", "", "g")
    img_fullname: ee.String = ee.String(tile_name).cat("_t").cat(ee.Date(start).millis().format())
        
    image: ee.Image = sdb.compute_inverse_depth(
                bounds=bounds,
                start=start,
                stop=stop,
                scale=tiler.zoom_to_scale(ee.Number.parse(tile.get("zoom"))).multiply(5), # scale to search for clean images
                missions=["S2", "L8"],
                filter_masked=True,
                skip_neighborhood_search=False
                # cloud_frequency_threshold_data=,
                # pansharpen=,
                # skip_scene_boundary_fix=,
                # bounds_buffer=
    ).clip(bounds)

    image = image.set(
        "fullname", img_fullname,
        "system:time_start", ee.Date(start).millis(),
        "system:time_stop", ee.Date(stop).millis(),
        "zoom", zoom,
        "tx", tx,
        "ty", ty
    )
    return image

def get_tile_intertidal_bathymetry(tile: ee.Feature, start: ee.String, stop: ee.String) -> ee.Image:
    """
    Get intertidal bathymetry based on tile geometry.
    Server-side compliant for GEE.

    args:
        tile (ee.Feature): tile geometry used to obtain bathymetry.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
    
    returns:
        ee.Image: image containing intertidal bathymetry covering tile.
    """

    bounds: ee.Geometry = ee.Feature(tile).geometry().bounds(1)
    sdb: Bathymetry = Bathymetry()
    zoom: ee.String = ee.String(tile.get("zoom"))
    tx: ee.String = ee.String(tile.get("tx"))
    ty: ee.String = ee.String(tile.get("ty"))
    tile_name: ee.String = ee.String("z").cat(zoom).cat("_x").cat(tx).cat("_y").cat(ty).replace("\.\d+", "", "g")
    img_fullname: ee.String = ee.String(tile_name).cat("_t").cat(ee.Date(start).millis().format())
        
    image: ee.Image = sdb.compute_intertidal_depth(
        bounds=bounds,
        start=start,
        stop=stop,
        scale=tiler.zoom_to_scale(ee.Number.parse(tile.get("zoom"))).multiply(5), # scale to search for clean images
        # missions=['S2', 'L8'],
        # filter: ee.Filter.dayOfYear(7*30, 9*30), # summer-only
        filter_masked=False, 
        # filterMaskedFraction = 0.5,
        # skip_scene_boundary_fix=False,
        # skip_neighborhood_search=False,
        neighborhood_search_parameters={"erosion": 0, "dilation": 0, "weight": 50},
        bounds_buffer=0,
        water_index_min=-0.05,
        water_index_max=0.15,
        # lowerCdfBoundary=45,
        # upperCdfBoundary=50,
        # cloud_frequency_threshold_data=0.15, 
        clip = True,
        mosaic_by_day = True
    )# .reproject(ee.Projection("EPSG:3857").atScale(90))

    image = image.set(
        "fullname", img_fullname,
        "system:time_start", ee.Date(start).millis(),
        "system:time_stop", ee.Date(stop).millis(),
        "zoom", zoom,
        "tx", tx,
        "ty", ty
    )

    return image

def tile_to_asset(
    image: ee.Image,
    tile: ee.Feature,
    export_scale: int,
    asset_path_prefix: str,
    asset_name: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    
    asset_id: str = f"{asset_path_prefix}/{asset_name}"
    asset: Dict[str, Any] = ee.data.getInfo(asset_id)
    if overwrite and asset:
        logger.info(f"deleting asset {asset}")
        ee.data.deleteAsset(asset_id)
    elif asset:
        logger.info(f"asset {asset} already exists, skipping {asset_name}")
        return
    task: ee.batch.Task = ee.batch.Export.image.toAsset(
        image,
        assetId=asset_id,
        description=asset_name,
        region=tile.geometry(),
        scale=export_scale,
        maxPixels= 1e10
    )
    task.start()
    logger.info(f"exporting {asset_name} to {asset_id}")

def tile_to_cloud_storage(
    image: ee.Image,
    tile: ee.Feature,
    crs: str,
    export_scale: int,
    bucket: str,
    bucket_path: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    with build('storage', 'v1') as storage:
        res = storage.objects().list(bucket=bucket, prefix="/".join(bucket_path.split("/")[:-1])).execute()
    if not overwrite:
        try:
            object_exists = any(map(lambda item: item.get("name").startswith(bucket_path), res.get("items")))
        except AttributeError:
            object_exists = False
        if object_exists:
            logger.info(f"object {bucket_path} already exists in bucket {bucket}, skipping")
            return
        
    task: ee.batch.Task = ee.batch.Export.image.toCloudStorage(
        image,
        bucket=bucket,
        description=bucket_path.replace("/", "_"),
        fileNamePrefix=bucket_path,
        region=tile.geometry(),
        scale=export_scale,
        crs=crs,
        fileFormat='GeoTIFF',
        formatOptions= {'cloudOptimized': True}, # enables easy QGIS plotting
        maxPixels= 1e10
    )
    task.start()
    return task

def metadata_to_cloud_storage(
    image: ee.Image,
    tile: ee.Feature,
    bucket: str,
    bucket_path: str,
    overwrite: bool
) -> Optional[ee.batch.Task]:
    with build('storage', 'v1') as storage:
        res = storage.objects().list(bucket=bucket, prefix="/".join(bucket_path.split("/")[:-1])).execute()
    if not overwrite:
        try:
            object_exists = any(map(lambda item: item.get("name").startswith(bucket_path), res.get("items")))
        except AttributeError:
            object_exists = False
        if object_exists:
            logger.info(f"object {bucket_path} already exists in bucket {bucket}, skipping")
            return

    meta_feature = ee.Feature(None, image.toDictionary().set("tx", tile.get("tx")).set("ty", tile.get("ty")))
        
    task: ee.batch.Task = ee.batch.Export.table.toCloudStorage(
        ee.FeatureCollection(meta_feature),
        bucket=bucket,
        description=bucket_path.replace("/", "_"),
        fileNamePrefix=bucket_path,
        fileFormat='csv',
        maxVertices=0
    )
    task.start()
    return task

def export_sdb_tiles(
    sink: str,
    tile_list: ee.List,
    num_tiles: int,
    export_scale: int,
    crs: str,
    sdb_tiles: ee.ImageCollection,
    name_suffix: str,
    mode: str,
    task_list: List[ee.batch.Task],
    overwrite: bool,
    bucket: Optional[str] = None
) -> List[ee.batch.Task]:
    """
    Export list of tiled images containing sub or intertidal tidal bathymetry. Fires off the tasks and adds to the list of tasks.
    based on: https://github.com/gee-community/gee_tools/blob/master/geetools/batch/imagecollection.py#L166

    args:
        sink (str): type of data sink to export to. Viable options are: "asset" and "cloud".
        tile_list (ee.List): list of tile features.
        num_tiles (int): number of tiles in `tile_list`.
        scale (int): scale of the export product.
        sdb_tiles (ee.ImageCollection): collection of subtidal bathymetry images corresponding
            to input tiles.
        name_suffix (str): unique identifier after tile statistics.
        task_list (List[ee.batch.Task]): list of tasks, adds tasks created to this list.
        overwrite (bool): whether to overwrite the current assets under the same `asset_path`.
        bucket (str): Bucket where the data is stored. Only used when sink = "cloud"
    
    returns:
        List[ee.batch.Task]: list of started tasks

    """
    if sink == "asset":
        user_name: str = ee.data.getAssetRoots()[0]["id"].split("/")[-1]
        asset_path_prefix: str = f"users/{user_name}/eo-bathymetry"
        ee.data.create_assets(asset_ids=[asset_path_prefix], asset_type="Folder", mk_parents=True)
    
    for i in range(num_tiles):
        # get tile
        temp_tile: ee.Feature = ee.Feature(tile_list.get(i))
        tile_metadata: Dict[str, Any] = temp_tile.getInfo()["properties"]
        tx: str = tile_metadata["tx"]
        ty: str = tile_metadata["ty"]
        zoom: str = tile_metadata["zoom"]
        # filter imagecollection based on tile
        filtered_ic: ee.ImageCollection = sdb_tiles \
            .filterMetadata("tx", "equals", tx) \
            .filterMetadata("ty", "equals", ty) \
            .filterMetadata("zoom", "equals", zoom)
        # if filtered correctly, only a single image remains
        img: ee.Image = ee.Image(filtered_ic.first())  # have to cast here
        img_name: str = sub(r"\.\d+", "", f"{mode}/z{zoom}/x{tx}/y{ty}/") + name_suffix 
        print("Submitting task for tile: ", img_name)
        # Export images
        if sink == "asset":  # Replace with case / switch in python 3.10
            task_img: Optional[ee.batch.Task] = tile_to_asset(
                image=img,
                tile=temp_tile,
                export_scale=export_scale,
                asset_path_prefix=asset_path_prefix,
                asset_name=img_name.replace("/","_"),
                overwrite=overwrite
            )
            if task_img: task_list.append(task_img)
        elif sink == "cloud":
            if not bucket:
                raise ArgumentError("Sink option requires \"bucket\" arg.")
            task_img: ee.batch.Task = tile_to_cloud_storage(
                image=img,
                tile=temp_tile,
                export_scale=export_scale,
                crs=crs, 
                bucket=bucket,
                bucket_path=img_name,
                overwrite=overwrite
            )

            task_meta: ee.batch.Task = metadata_to_cloud_storage(
                image=img,
                tile=temp_tile,
                bucket=bucket,
                bucket_path=sub(r"\.\d+", "", f"{mode}_meta/z{zoom}/x{tx}/y{ty}/") + name_suffix,
                overwrite=overwrite
            )
        else:
            raise ArgumentError("unrecognized data sink: {sink}")
        task_list.append(task_img)
        task_list.append(task_meta)
    return task_list

def export_tiles(
    sink: str,
    mode: str,
    geometry: ee.Geometry,
    zoom: int,
    start: str,
    stop: str,
    scale: Optional[float] = None,
    crs: str = "EPSG:4326",
    buf_pix: int = 0,
    step_months: int = 3,
    window_months: int = 24,
    overwrite: bool = False,
    bucket: Optional[str] = None
) -> None:
    """
    From a geometry, creates tiles of input zoom level, calculates subtidal bathymetry in those
    tiles, and exports those tiles.

    args:
        sink (str): type of data sink to export to. Viable options are: "asset" and "cloud".
        mode (str): either "subtidal" or "intertidal" for select type of bathymetry to export.
        geometry (ee.Geometry): geometry of the area of interest.
        zoom (int): zoom level of the to-be-exported tiles.
        start (ee.String): start date in YYYY-MM-dd format.
        stop (ee.String): stop date in YYYY-MM-dd format.
        scale Optional(float): scale of the product to be exported. Defaults tiler.zoom_to_scale(zoom).getInfo().
        crs (str): projection of the output image.
        buf_pix (int): buffer around the tile (in pixels).
        step_months (int): steps with which to roll the window over which the subtidal bathymetry
            is calculated.
        windows_months (int): number of months over which the bathymetry is calculated.
    """

    def create_year_window(year: ee.Number, month: ee.Number) -> ee.Dictionary:
        t: ee.Date = ee.Date.fromYMD(year, month, 1)
        d_format: str = "YYYY-MM-dd"
        return ee.Dictionary({
            "start": t.format(d_format),
            "stop": t.advance(window_months, 'month').format(d_format)
            })
    
    window_length: int = (parse(stop).year-parse(start).year)*12+(parse(stop).month-parse(start).month) # in months
    dates: ee.List = ee.List.sequence(parse(start).year, parse(stop).year-window_months/12).map(
        lambda year: ee.List.sequence(1, None, step_months, int((window_length-window_months)/step_months)+1).map(partial(create_year_window, year))
    ).flatten() # NOTE, still buggy, works for yearly composites. Not nice for end_date "2022-03-01"; error Date.fromYMD: Bad year/month/day: 2021/13/1.

    #dates = ee.List([dates.get(0)]) #ADJUSTED TO SELECT FIRST DATE ONLY
    
    # Get tiles
    #tiled: ee.FeatureCollection = tiler.get_tiles_for_geometry(geometry, ee.Number(zoom))
    #tiles: ee.FeatureCollection = tiled.map(lambda tile: tile.buffer(buf_pix*scale/111120, ee.ErrorMargin((buf_pix*scale*0.01)/111120, 'projected'), proj="EPSG:4326")) # fix for unequal buffer in EPSG3857: https://www.reddit.com/r/QGIS/comments/oo1jgh/buffer_points_on_epsg_4326_wgs_84/
    #tiledb: ee.FeatureCollection = tiled.map(lambda tile: tile.buffer(buf_pix*scale/111120, ee.ErrorMargin((buf_pix*scale*0.01)/111120, 'projected'), proj="EPSG:4326")) # ADJUSTED TO SELECT SINGLE TILE & fix for unequal buffer in EPSG3857: https://www.reddit.com/r/QGIS/comments/oo1jgh/buffer_points_on_epsg_4326_wgs_84/
    #tile: ee.Feature = ee.Feature(tiledb.filterMetadata("tx", "equals", tx_in).filterMetadata("ty", "equals", ty_in).first()) #11 #ADJUSTED TO SELECT SINGLE TILE
    tile: ee.Feature =  ee.Feature(geometry.buffer(5*scale/111120, ee.ErrorMargin((buf_pix*scale*0.01)/111120, 'projected'), proj="EPSG:4326"))
    tiles: ee.FeatureCollection = ee.FeatureCollection(tile) #ADJUSTED TO SELECT SINGLE TILE

    if scale == None: scale: float = tiler.zoom_to_scale(zoom).getInfo() # not specified, defaults to pre-set float
    else: scale: scale # specified
    task_list: List[ee.batch.Task] = []
    #num_tiles: int = tiles.size().getInfo()
    #tile_list: ee.List = tiles.toList(num_tiles)

    for date in dates.getInfo():
        print(date)
        if "subtidal" in mode:
            sdb_tiles: ee.ImageCollection = tiles.map(
                lambda tile: get_tile_subtidal_bathymetry(
                    tile=tile,
                    start=ee.String(date["start"]),
                    stop=ee.String(date["stop"])
                )#.clip(geometry) # clip individual tiles to match geometry of aoi
            )
        
        elif "intertidal" in mode:

            # filter tiles on possibility of having GTSM data. NOTE, the temporal extend takes the entire period of the composite, not the individual images. 
            # it is assumed if theres GTSM data, we will always find an image that is closeby. This assumption might not hold for all cases, investigate in the future (TODO)
            tiles = tiles.map(lambda tile: filter_tiles_close_to_GTSM(gtsm_col, tile=tile, start_date=ee.String(date["start"]), stop_date=ee.String(date["stop"])))
            tiles = tiles.filter(ee.Filter.gt('max_gtsm_size', 0))

            sdb_tiles: ee.ImageCollection = tiles.map(
                lambda tile: get_tile_intertidal_bathymetry(
                    tile=tile,
                    start=ee.String(date["start"]),
                    stop=ee.String(date["stop"])
                )#.clip(geometry)#.select('ndwi').rename('water_score') # clip individual tiles to match geometry of aoi, select ndwi and rename
            )

            # only export the ones with coupled GTSM data (i.e. depth values iso proxies)
            # NOTE, since we already filtered tiles above, we assume the code below is not needed anymore as sdb_tiles_up would be the same as sdb_tiles, investigate in the future (TODO)
            # if not the case, we might catch this by adding a SkipEmptyTiles in the batch image export function 
            #sdb_tile_prop: ee.FeatureCollection = sdb_tiles.select("gtsm_gebco_data_allempty") # ADDED filter GTSM data availability property
            #innerJoin = ee.Join.inner('primary', 'secondary') # ADDED construct join function
            #joined_tiles = innerJoin.apply(tiles, sdb_tile_prop, ee.Filter.equals(leftField= "id", rightField= "id")) # ADDED join tiles & sdb_tiles on id
            #joined_tiles_clean = joined_tiles.map(lambda feature: ee.Feature(feature.get('primary')).copyProperties(feature.get('secondary'))) # ADDED clean up joined tiles, keeping the tiles and only adding the selected property
            #tile_list: ee.FeatureCollection  = joined_tiles_clean.filterMetadata("gtsm_gebco_data_allempty", "equals", False) #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES
            #sdb_tiles_up: ee.ImageCollection = sdb_tiles.filterMetadata("gtsm_gebco_data_allempty", "equals", False) #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES

        num_tiles: int = tiles.size().getInfo() # tile_list #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES
        #return sdb_tiles, tiles, tile_list#, sdb_tiles_up
        if num_tiles > 0:  #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES
            tile_list: ee.List = tiles.toList(num_tiles)  # tile_list_up: ee.List = tile_list.toList(num_tiles) #ADDED TO UPDATE EXPORT FOR ONLY CALIBRATED IMAGES

            #return sdb_tiles, tiles, tile_list#, sdb_tiles_up, tile_list_up

            # Now export tiles
            task_list = export_sdb_tiles(
                sink=sink,
                tile_list=tile_list, # tile_list_up
                num_tiles=num_tiles,
                mode=mode,
                export_scale=scale,
                crs=crs,
                sdb_tiles=sdb_tiles, # sdb_tiles_up
                name_suffix=f"t{date['start']}_{date['stop']}_{scale}m",
                task_list=task_list,
                overwrite=overwrite,
                bucket=bucket
            )

            # return task_list # toggle off when you need more dates to be run..

In [21]:
# export bathymetry based on standardized tiling practice [OLD!!!]
# allfiles = os.listdir(os.path.join(main_fol, aoi_fol))
# tasks_lists = []
# for i in allfiles:
#     if i.endswith("geojson"):# and not "CMEMS" in i and ("GER" in i):# or "BRA" in i or "ZAF" in i):
#         print(i)
#         with open(os.path.join(main_fol, aoi_fol,i)) as f:
#             AoIjson = geojson.load(f)
#         # try: # drawn polygon in this script
#         #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"]) 
#         # except: # drawn in QGIS / ArcGIS and written to geojson there (client file)
#         #     AoI = Polygon(AoIjson["features"][0]["geometry"]["coordinates"][0])
#         # bounds = ee.Geometry.Polygon([[[a,b] for a, b in zip(*AoI.exterior.coords.xy)]])
#         bounds = ee.Geometry.MultiPolygon(AoIjson["features"][0]["geometry"]["coordinates"])

#         if "intertidal" in mode:
#             # make the AoI a gdf dataframe (takes about 5 min)
#             gdf_AOI = gpd.GeoDataFrame.from_features(AoIjson)

#             # retrieve the (buffered) global coastal intertidal mask (geojson), using parquet
#             maskfiles = os.listdir(main_fol / mask_fol / "2024")#r"2023\Buffered\AOI_results")
#             for j in maskfiles:
#                 if '.parquet' in j:
#                     print(j)
#                     # read parquet file
#                     df = gpd.read_parquet(main_fol / mask_fol / "2024" / j)#r"2023\Buffered\AOI_results" / j)

#                     # inner join to match the AOI and mask
#                     gdfs = df.sjoin(gdf_AOI, how="inner")

#                     # make a multipolygon
#                     gdfs_mp = gpd.GeoDataFrame({'geometry': [MultiPolygon(list(gdfs.geometry))]})

#                     # make a ee.Geometry.MultiPolygon to have bounds
#                     data = json.loads(gdfs_mp.to_json())
#                     feature_collection = geojson.FeatureCollection(data['features'])
#                     bounds = ee.Geometry.MultiPolygon(feature_collection["features"][0]["geometry"]["coordinates"])

#         #sdb_tiles, tiles, tile_list = export_tiles(....)
#         task_list = export_tiles(sink="cloud", mode=mode, geometry=bounds, zoom=zoomed_list[sel_tile], start=start_date, stop=stop_date, 
#                      crs=crs, scale=scale, buf_pix=5, step_months=compo_int, window_months=compo_len, overwrite=True, bucket=bucket)
#         tasks_lists.append(task_list)

In [103]:
# project_name = "AOI_NL" # name of the project AoI, or one in the folder
# start_date = "2021-01-01" # start date of the composites
# stop_date = "2022-01-01" # end date of the composites
# compo_int = 3 # composite interval [months]
# compo_len = 3 # composite length [months]
# scale = 100  # output resolution of the image [m]

In [25]:
df_boxes["AOI"]

0    AOI_AUS.geojson
1     AOI_NL.geojson
2     AOI_NL.geojson
3     AOI_NL.geojson
4     AOI_NL.geojson
Name: AOI, dtype: object

In [26]:
# compute bathy and export to GCS: run over the single tiles; world, multiple polygons or single polygon
# when run submitted, check task progress at: https://code.earthengine.google.com/tasks or https://console.cloud.google.com/earth-engine/tasks?project=bathymetry

allboxes = os.listdir(os.path.join(main_fol, box_fol)) # query all info in the dedicated folder
sp = 1 # single polygon run (1) or multiple polygons run (0)

# pre-configuring the tiles to compute
if ".csv" in allboxes: # csv present, run over tiles by name in csv file

    # file deviates from the 8th column onwards from polygon cases (there's no AOI)
    print("A dataframe with the boxes enabled allows to run over the world, do you want to proceed? Enable this here...")
    df_boxrun = pd.read_csv(main_fol / box_fol / "df_boxes_world_Z%s.csv"%(zoomed_list[sel_tile]))

    # sort if needed
    # ...

else: # no csv, run over tiles by means of looping over names

    if sp == 1: # single polygon run
        df_boxrun = df_boxes[df_boxes["AOI"] == project_name + ".geojson"] # filter on the project name

    elif sp == 0: # multiple polygons run
        df_boxrun = df_boxes.drop_duplicates(subset=["name"], keep="first") # remove duplicate rows

    # filtering on specific tiles
    print(df_boxrun)


        
# run over the tiles 
# task_list = [] # save the tasked tile progress..
# for idx, i in df_boxrun.reset_index().iterrows():
#     print(i["name"])

#     # filter specific tile
#     #if "1065" in i.tx and "660 in i.ty:

#     # break on certain tile
#     # if idx == 10:
#     #     break

#     # get AoI
#     with open(i.path) as f:
#         AoIjson = geojson.load(f)

#     # reconstruct geometry 
#     # NOTE, all is the same except for the id in the feature that does not come through
#     feat = ee.Geometry.Polygon(AoIjson["features"][0]["geometry"]["coordinates"], AoIjson["crs"]['properties']["name"], False)
#     prop = {"tx": ee.String(str(AoIjson["features"][0]["properties"]["tx"])), "ty": ee.String(str(AoIjson["features"][0]["properties"]["ty"])), "zoom": ee.String(str(AoIjson["features"][0]["properties"]["zoom"]))}
#     feat = ee.Feature(feat).set(prop)

#     task_list.append(export_tiles(sink="cloud", mode=mode, geometry=feat, zoom=zoomed_list[sel_tile], start=start_date, stop="2022-01-01", 
#                     crs=crs, scale=scale, buf_pix=5, step_months=compo_int, window_months=compo_len, overwrite=True, bucket=bucket))

            name id     tx     ty  zoom  \
1  z10_x527_y331  0  527.0  331.0    10   
2  z10_x527_y332  1  527.0  332.0    10   
3  z10_x528_y331  3  528.0  331.0    10   
4  z10_x528_y332  4  528.0  332.0    10   

                                                path             AOI  \
1  p:\11209821-cmems-global-sdb\00_miscellaneous\...  AOI_NL.geojson   
2  p:\11209821-cmems-global-sdb\00_miscellaneous\...  AOI_NL.geojson   
3  p:\11209821-cmems-global-sdb\00_miscellaneous\...  AOI_NL.geojson   
4  p:\11209821-cmems-global-sdb\00_miscellaneous\...  AOI_NL.geojson   

  ref_region                                           geometry area_perc_org  \
1        NEU  POLYGON ((587036.377 7083572.286, 626172.136 7...         39.49   
2        NEU  POLYGON ((587036.377 7044436.527, 626172.136 7...         57.48   
3        NEU  POLYGON ((626172.136 7083572.286, 665307.894 7...         40.07   
4        NEU  POLYGON ((626172.136 7044436.527, 665307.894 7...          0.99   

  area_perc_red  

In [27]:
df_boxrun

,geometry,name,id,tx,ty,zoom,path,AOI
1,"POLYGON ((587036.377 7044436.527, 626172.136 7...",z10_x527_y331,0,527.0,331.0,10,p:\11209821-cmems-global-sdb\00_miscellaneous\...,AOI_NL.geojson
2,"POLYGON ((587036.377 7005300.769, 626172.136 7...",z10_x527_y332,1,527.0,332.0,10,p:\11209821-cmems-global-sdb\00_miscellaneous\...,AOI_NL.geojson
3,"POLYGON ((626172.136 7044436.527, 665307.894 7...",z10_x528_y331,3,528.0,331.0,10,p:\11209821-cmems-global-sdb\00_miscellaneous\...,AOI_NL.geojson
4,"POLYGON ((626172.136 7005300.769, 665307.894 7...",z10_x528_y332,4,528.0,332.0,10,p:\11209821-cmems-global-sdb\00_miscellaneous\...,AOI_NL.geojson


In [25]:
tasks = []
for i in task_list:
    if i is not None:
        tasks.append(i)

len(tasks)

150

In [32]:
tasks

[[<Task Y4BBROCEES3KUCCWVWZOEZRZ Type.EXPORT_IMAGE: intertidal_improved_100m_upscaled_z10_x485_y375_t2021-01-01_2022-01-01_100m (State.UNSUBMITTED)>,
  <Task 5DXMMOBFKAPPAJXXLXHZSPPB Type.EXPORT_TABLE: intertidal_improved_100m_upscaled_meta_z10_x485_y375_t2021-01-01_2022-01-01_100m (State.UNSUBMITTED)>],
 [<Task BEZTQ74WGTTRPKHPUUE3WNQI Type.EXPORT_IMAGE: intertidal_improved_100m_upscaled_z10_x485_y376_t2021-01-01_2022-01-01_100m (State.UNSUBMITTED)>,
  <Task 72SPOIUZZ5BI6KONWZHF7REM Type.EXPORT_TABLE: intertidal_improved_100m_upscaled_meta_z10_x485_y376_t2021-01-01_2022-01-01_100m (State.UNSUBMITTED)>],
 [<Task IWPJY7DQZPJ5Q4ATWHYPBA4N Type.EXPORT_IMAGE: intertidal_improved_100m_upscaled_z10_x485_y389_t2021-01-01_2022-01-01_100m (State.UNSUBMITTED)>,
  <Task TRLZIGLNKKCXMFMZJLBCGUOC Type.EXPORT_TABLE: intertidal_improved_100m_upscaled_meta_z10_x485_y389_t2021-01-01_2022-01-01_100m (State.UNSUBMITTED)>],
 [<Task DPVE5QH2P26AK6STEIMQGVWL Type.EXPORT_IMAGE: intertidal_improved_100m_upsca

In [28]:
len(tasks[0])

2

In [29]:
# save the task list as a pickle file
import pickle
with open(main_fol / prog_fol / ("tasks_" + project_name +".pkl"), "wb") as f:
    pickle.dump(task_list, f)

In [30]:
# open the task list as a pickle file
with open(main_fol / prog_fol / ("tasks_" + project_name +".pkl"), "rb") as f:
    task_list2 = pickle.load(f)

task_list2[1][0].status()

{'state': 'COMPLETED',
 'description': 'intertidal_improved_100m_upscaled_z10_x485_y375_t2021-01-01_2022-01-01_100m',
 'creation_timestamp_ms': 1731999740919,
 'update_timestamp_ms': 1732000315044,
 'start_timestamp_ms': 1731999749360,
 'task_type': 'EXPORT_IMAGE',
 'destination_uris': ['https://console.developers.google.com/storage/browser/cmems-sdb/intertidal_improved_100m_upscaled/z10/x485/y375/'],
 'attempt': 1,
 'batch_eecu_usage_seconds': 3673.1240234375,
 'id': 'Y4BBROCEES3KUCCWVWZOEZRZ',
 'name': 'projects/bathymetry/operations/Y4BBROCEES3KUCCWVWZOEZRZ'}

In [31]:
len(task_list2)

175

## [Optional] Download the TIFFs from the cloud (also present in post-processing scripts)

In [35]:
# store locally (from GCS) to visualize in QGIS / ArcGIS (can also download manually via Cloud Storage platform)

# create or check if local storage folder is present
if not os.path.exists(os.path.join(main_fol, output_fol)):
    os.makedirs(os.path.join(main_fol, output_fol))

# get file names
client = storage.Client()
ls = [blob for blob in client.list_blobs(bucket)] 

# downloading composites to a local folder while only keeping subtidal / intertidal folder (i.e. other nested folders are flattened)
check_files = []
for blob in ls:
    #if "tif" in blob.name:
    mode_fol = blob.name.split('/')[0]
    zoom_level = blob.name.split('/')[1]
    try: zoom_level_num = zoom_level.split("z")[1]
    except: zoom_level_num = zoom_level.split("z")[0]
    if mode_fol == mode and str(zoomed_list[sel_tile]) == zoom_level_num:
        file_name = "_".join(blob.name.split('/')[1:])
        check_files.append(file_name) #blob.name.split('/')[-1]
        if not os.path.exists(os.path.join(main_fol, output_fol, mode_fol)): # create subfolder
            os.makedirs(os.path.join(main_fol, output_fol, mode_fol))
        blob.download_to_filename(os.path.join(main_fol, output_fol, mode_fol, file_name))
        print('Stored: ', file_name) # check progress

# elaborate on possibility of storing locally
if len(check_files) == 0:
    print('Please enable GCS storeing of images first, before toggling on local storage option')

Stored:  z9_x113_y116_t2021-01-01_2022-01-01.tif
Stored:  z9_x113_y117_t2021-01-01_2022-01-01.tif
Stored:  z9_x114_y116_t2021-01-01_2022-01-01.tif
Stored:  z9_x114_y117_t2021-01-01_2022-01-01.tif
Stored:  z9_x114_y118_t2021-01-01_2022-01-01.tif
Stored:  z9_x115_y116_t2021-01-01_2022-01-01.tif
Stored:  z9_x115_y117_t2021-01-01_2022-01-01.tif
Stored:  z9_x115_y118_t2021-01-01_2022-01-01.tif
Stored:  z9_x163_y228_t2021-01-01_2022-01-01.tif
Stored:  z9_x163_y229_t2021-01-01_2022-01-01.tif
Stored:  z9_x164_y228_t2021-01-01_2022-01-01.tif
Stored:  z9_x164_y229_t2021-01-01_2022-01-01.tif
Stored:  z9_x189_y291_t2021-01-01_2022-01-01.tif
Stored:  z9_x189_y292_t2021-01-01_2022-01-01.tif
Stored:  z9_x190_y291_t2021-01-01_2022-01-01.tif
Stored:  z9_x190_y292_t2021-01-01_2022-01-01.tif
Stored:  z9_x263_y160_t2021-01-01_2022-01-01.tif
Stored:  z9_x263_y161_t2021-01-01_2022-01-01.tif
Stored:  z9_x263_y162_t2021-01-01_2022-01-01.tif
Stored:  z9_x264_y160_t2021-01-01_2022-01-01.tif
Stored:  z9_x264_y16